In [1]:
import os
import json
import shutil
from collections import defaultdict
from tqdm import tqdm

DATASET_PATH = "/kaggle/input/datasets/sonlest/bom-dataset-v2/BOM-Dataset-v2/annotations"
ANNOTATION_PATH = os.path.join(DATASET_PATH, "instances.json")

OUTPUT_PATH = "/kaggle/working/BOM-Dataset-Detectron2"
OUTPUT_JSON = os.path.join(OUTPUT_PATH, "annotations_v1.json")

os.makedirs(OUTPUT_PATH, exist_ok=True)

In [2]:
def load_dataset(json_path):
    with open(json_path, 'r') as f:
        data = json.load(f)
    return data

In [3]:
def fix_categories(data):
    categories = data["categories"]

    print("\n🔧 FIXING CATEGORY IDs...")

    # Map old_id -> new_id
    id_map = {}
    for new_id, cat in enumerate(categories):
        id_map[cat["id"]] = new_id
        cat["id"] = new_id

    # Update annotations
    for ann in data["annotations"]:
        ann["category_id"] = id_map[ann["category_id"]]

    print("✅ Category IDs fixed:", id_map)
    return data

In [4]:
def fix_bboxes(data):
    print("\n🔧 FIXING BBOXES...")

    image_map = {img["id"]: img for img in data["images"]}
    valid_annotations = []

    removed = 0
    fixed = 0

    for ann in data["annotations"]:
        img = image_map[ann["image_id"]]
        x, y, w, h = ann["bbox"]

        if w <= 0 or h <= 0:
            removed += 1
            continue

        # Clamp bbox
        x = max(0, x)
        y = max(0, y)

        if x + w > img["width"]:
            w = img["width"] - x
            fixed += 1

        if y + h > img["height"]:
            h = img["height"] - y
            fixed += 1

        ann["bbox"] = [x, y, w, h]
        ann["area"] = w * h

        valid_annotations.append(ann)

    data["annotations"] = valid_annotations

    print(f"✅ Fixed bbox: {fixed}")
    print(f"❌ Removed invalid bbox: {removed}")

    return data

In [5]:
def remove_empty_images(data):
    print("\n🧹 REMOVING EMPTY IMAGES...")

    ann_image_ids = set(ann["image_id"] for ann in data["annotations"])

    valid_images = []
    removed = 0

    for img in data["images"]:
        if img["id"] in ann_image_ids:
            valid_images.append(img)
        else:
            removed += 1

    data["images"] = valid_images

    print(f"❌ Removed images without annotations: {removed}")
    return data

In [6]:
def validate_dataset(data):
    print("\n📊 VALIDATING DATASET...")

    errors = []

    cat_ids = set(cat["id"] for cat in data["categories"])

    for ann in data["annotations"]:
        if ann["category_id"] not in cat_ids:
            errors.append("Invalid category_id")

        if len(ann["bbox"]) != 4:
            errors.append("Invalid bbox format")

    if len(errors) == 0:
        print("✅ Dataset is VALID for Detectron2")
    else:
        print("❌ Errors found:", set(errors))

In [7]:
def copy_images(data):
    print("\n📂 COPYING IMAGES...")

    image_dir = os.path.join(DATASET_PATH, "images")
    output_image_dir = os.path.join(OUTPUT_PATH, "images")

    os.makedirs(output_image_dir, exist_ok=True)

    for img in tqdm(data["images"]):
        src = os.path.join(image_dir, img["file_name"])
        dst = os.path.join(output_image_dir, img["file_name"])

        if os.path.exists(src):
            shutil.copy(src, dst)

In [8]:
def save_dataset(data):
    print("\n💾 SAVING DATASET...")

    with open(OUTPUT_JSON, "w") as f:
        json.dump(data, f)

    print("✅ Saved to:", OUTPUT_PATH)

In [9]:
def main():
    print("🚀 START DETECTRON2 DATASET PIPELINE")

    data = load_dataset(ANNOTATION_PATH)

    data = fix_categories(data)
    data = fix_bboxes(data)
    data = remove_empty_images(data)

    validate_dataset(data)

    copy_images(data)
    save_dataset(data)

    print("\n🎯 DONE! DATASET READY FOR DETECTRON2")

if __name__ == "__main__":
    main()

🚀 START DETECTRON2 DATASET PIPELINE

🔧 FIXING CATEGORY IDs...
✅ Category IDs fixed: {1: 0, 2: 1, 3: 2}

🔧 FIXING BBOXES...
✅ Fixed bbox: 0
❌ Removed invalid bbox: 0

🧹 REMOVING EMPTY IMAGES...
❌ Removed images without annotations: 0

📊 VALIDATING DATASET...
✅ Dataset is VALID for Detectron2

📂 COPYING IMAGES...


100%|██████████| 171/171 [00:00<00:00, 72859.20it/s]


💾 SAVING DATASET...
✅ Saved to: /kaggle/working/BOM-Dataset-Detectron2

🎯 DONE! DATASET READY FOR DETECTRON2
